# マケインの聖地が多すぎる！ グラフ理論で最短ルートを見つけよう

## 出発地、目的地、および経由地の定義

In [1]:
"""テスト用の出発地、目的地、および経由地の定義"""
import os
import math
from typing import cast
from itertools import permutations
import requests

from data_model import Spot,TravelMode
from main import coordinates_from_google_maps
from main import generate_google_maps_url
from main import build_distance_matrix

# 出発地
origin_point = Spot("豊橋駅", "https://maps.app.goo.gl/QJnQT8GWxRBJuZ9o9", "🛫")

# 目的地
destination_point = Spot(
    "豊橋駅前展望会議室", "https://maps.app.goo.gl/RYHLQdmArn24w1n68","🏢"
)

# 経由地
# <https://makeine-anime.com/special/map/>
waypoints = [
    Spot("精文館書店豊橋本店", "https://maps.app.goo.gl/rMin692aMHFGpmh2A", "①"),  # ①
    Spot(
        "豊鉄市内線駅前停留所",
        "https://maps.app.goo.gl/n8gXQQecdM7HeEQCA",
        "②"
    ),  # ②
    Spot("駅ビルカフェUNO-UNO", "https://maps.app.goo.gl/5M2Rp5CwViWeReiY8", "⑦"),  # ⑦
    Spot("ボン.千賀", "https://maps.app.goo.gl/QpT13VUjrixQGJr16", "⑧"),  # ⑧
    Spot("豊橋市まちなか図書館", "https://maps.app.goo.gl/8gDMzz9NHTixLS7T7", "⑨"),  # ⑨
    Spot("豊橋駅ビルカルミア", "https://maps.app.goo.gl/kZeRMHnt8GzGd6ti8", "⑩"),  # ⑩
    Spot("むらたのたこやき", "https://maps.app.goo.gl/KFPCCbEoKu5PPkU26", "⑪"),  # ⑪
    Spot("豊川堂カルミヤ店", "https://maps.app.goo.gl/74gCMRktKUZKJgk37", "⑫"),  # ⑫
    Spot("水上ビル", "https://maps.app.goo.gl/GcT6SuSYZtdVYWgq8", "⑲"), # ⑲
]
# Google Maps は最大で 9 個の経由地を追加できます
# Spot("珈琲とカヌレ", "https://maps.app.goo.gl/NXUCuZ1Ty2CErXLR6"),  # ⑳

## List での順番で Google Maps の URL を生成

In [2]:
GOOGLE_MAP_URL = generate_google_maps_url(
    origin_point, destination_point, waypoints, TravelMode.WALKING
)
print("Google Maps の URL \n", GOOGLE_MAP_URL)

Google Maps の URL 
 https://www.google.com/maps/dir/?api=1&origin=%E8%B1%8A%E6%A9%8B%E9%A7%85&destination=%E8%B1%8A%E6%A9%8B%E9%A7%85%E5%89%8D%E5%B1%95%E6%9C%9B%E4%BC%9A%E8%AD%B0%E5%AE%A4&travelmode=walking&waypoints=%E7%B2%BE%E6%96%87%E9%A4%A8%E6%9B%B8%E5%BA%97%E8%B1%8A%E6%A9%8B%E6%9C%AC%E5%BA%97%7C%E8%B1%8A%E9%89%84%E5%B8%82%E5%86%85%E7%B7%9A%E9%A7%85%E5%89%8D%E5%81%9C%E7%95%99%E6%89%80%7C%E9%A7%85%E3%83%93%E3%83%AB%E3%82%AB%E3%83%95%E3%82%A7UNO-UNO%7C%E3%83%9C%E3%83%B3.%E5%8D%83%E8%B3%80%7C%E8%B1%8A%E6%A9%8B%E5%B8%82%E3%81%BE%E3%81%A1%E3%81%AA%E3%81%8B%E5%9B%B3%E6%9B%B8%E9%A4%A8%7C%E8%B1%8A%E6%A9%8B%E9%A7%85%E3%83%93%E3%83%AB%E3%82%AB%E3%83%AB%E3%83%9F%E3%82%A2%7C%E3%82%80%E3%82%89%E3%81%9F%E3%81%AE%E3%81%9F%E3%81%93%E3%82%84%E3%81%8D%7C%E8%B1%8A%E5%B7%9D%E5%A0%82%E3%82%AB%E3%83%AB%E3%83%9F%E3%83%A4%E5%BA%97%7C%E6%B0%B4%E4%B8%8A%E3%83%93%E3%83%AB


## Google Maps の URL から GPS の座標を取得

In [3]:
# 経由地の座標を取得
for spot in waypoints:
    spot.coordinates = coordinates_from_google_maps(spot.maps_url)

# 出発点と目的地の座標を取得
origin_point.coordinates = coordinates_from_google_maps(origin_point.maps_url)
destination_point.coordinates = coordinates_from_google_maps(destination_point.maps_url)

## 出発点と到着点は決めた場合（最近傍法）

In [4]:
all_spots = [origin_point] + waypoints + [destination_point]
all_distance_matrix = build_distance_matrix(all_spots)

# 出発点と目的地を除いた未訪問スポットの集合
unvisited = set(range(1, len(all_spots) - 1))

route_indices = [0]
# 現在のスポットのインデックス、最初は出発点（0）
current = 0

In [5]:
while unvisited:
    print(f"\n現在のスポット: {all_spots[current].label} {all_spots[current].name}")

    # 未訪問スポットを現在地から近い順に表示
    candidates = sorted(
        unvisited,
        key=lambda index, current=current: all_distance_matrix[current][index],
    )
    print("候補スポット（近い順）:")
    for index in candidates:
        distance = all_distance_matrix[current][index]
        print(f"  {all_spots[index].label} {all_spots[index].name}: {distance:.2f} m")

    next_index = candidates[0]
    next_distance = all_distance_matrix[current][next_index]
    print(f"次のスポット： {all_spots[next_index].label} {all_spots[next_index].name}（{next_distance:.2f} m）")
    route_indices.append(next_index)
    unvisited.remove(next_index)
    current = next_index



現在のスポット: 🛫 豊橋駅
候補スポット（近い順）:
  ⑫ 豊川堂カルミヤ店: 65.37 m
  ⑪ むらたのたこやき: 70.76 m
  ⑩ 豊橋駅ビルカルミア: 77.36 m
  ⑦ 駅ビルカフェUNO-UNO: 96.96 m
  ② 豊鉄市内線駅前停留所: 152.36 m
  ① 精文館書店豊橋本店: 246.91 m
  ⑧ ボン.千賀: 297.50 m
  ⑨ 豊橋市まちなか図書館: 490.13 m
  ⑲ 水上ビル: 501.05 m
次のスポット： ⑫ 豊川堂カルミヤ店（65.37 m）

現在のスポット: ⑫ 豊川堂カルミヤ店
候補スポット（近い順）:
  ⑦ 駅ビルカフェUNO-UNO: 35.44 m
  ⑩ 豊橋駅ビルカルミア: 39.73 m
  ⑪ むらたのたこやき: 47.48 m
  ② 豊鉄市内線駅前停留所: 97.06 m
  ① 精文館書店豊橋本店: 190.93 m
  ⑧ ボン.千賀: 232.97 m
  ⑨ 豊橋市まちなか図書館: 433.20 m
  ⑲ 水上ビル: 448.16 m
次のスポット： ⑦ 駅ビルカフェUNO-UNO（35.44 m）

現在のスポット: ⑦ 駅ビルカフェUNO-UNO
候補スポット（近い順）:
  ⑩ 豊橋駅ビルカルミア: 35.04 m
  ⑪ むらたのたこやき: 49.06 m
  ② 豊鉄市内線駅前停留所: 62.17 m
  ① 精文館書店豊橋本店: 155.54 m
  ⑧ ボン.千賀: 208.80 m
  ⑨ 豊橋市まちなか図書館: 419.37 m
  ⑲ 水上ビル: 438.33 m
次のスポット： ⑩ 豊橋駅ビルカルミア（35.04 m）

現在のスポット: ⑩ 豊橋駅ビルカルミア
候補スポット（近い順）:
  ⑪ むらたのたこやき: 14.14 m
  ② 豊鉄市内線駅前停留所: 75.86 m
  ① 精文館書店豊橋本店: 169.82 m
  ⑧ ボン.千賀: 242.35 m
  ⑨ 豊橋市まちなか図書館: 454.41 m
  ⑲ 水上ビル: 473.10 m
次のスポット： ⑪ むらたのたこやき（14.14 m）

現在のスポット: ⑪ むらたのたこやき
候補スポット（近い順）:
  ② 豊鉄市内線駅前停留所: 86.38 m
  ① 精

In [6]:
route_indices.append(len(all_spots) - 1)

route_spots = [all_spots[i] for i in route_indices]

print("\n最適ルート：")
for spot in route_spots:
    print(f"{spot.label} {spot.name}")

route_url = generate_google_maps_url(
    origin=route_spots[0],
    destination=route_spots[-1],
    waypoints=route_spots[1:-1],
    travel_mode=TravelMode.WALKING,
)

print("\nGoogle Maps の URL:")
print(route_url)

# 総距離を計算
route_total_distance = sum(
    all_distance_matrix[route_indices[i]][route_indices[i + 1]]
    for i in range(len(route_indices) - 1)
)

print(f"\n総距離: {route_total_distance:.2f} m")


最適ルート：
🛫 豊橋駅
⑫ 豊川堂カルミヤ店
⑦ 駅ビルカフェUNO-UNO
⑩ 豊橋駅ビルカルミア
⑪ むらたのたこやき
② 豊鉄市内線駅前停留所
① 精文館書店豊橋本店
⑧ ボン.千賀
⑨ 豊橋市まちなか図書館
⑲ 水上ビル
🏢 豊橋駅前展望会議室

Google Maps の URL:
https://www.google.com/maps/dir/?api=1&origin=%E8%B1%8A%E6%A9%8B%E9%A7%85&destination=%E8%B1%8A%E6%A9%8B%E9%A7%85%E5%89%8D%E5%B1%95%E6%9C%9B%E4%BC%9A%E8%AD%B0%E5%AE%A4&travelmode=walking&waypoints=%E8%B1%8A%E5%B7%9D%E5%A0%82%E3%82%AB%E3%83%AB%E3%83%9F%E3%83%A4%E5%BA%97%7C%E9%A7%85%E3%83%93%E3%83%AB%E3%82%AB%E3%83%95%E3%82%A7UNO-UNO%7C%E8%B1%8A%E6%A9%8B%E9%A7%85%E3%83%93%E3%83%AB%E3%82%AB%E3%83%AB%E3%83%9F%E3%82%A2%7C%E3%82%80%E3%82%89%E3%81%9F%E3%81%AE%E3%81%9F%E3%81%93%E3%82%84%E3%81%8D%7C%E8%B1%8A%E9%89%84%E5%B8%82%E5%86%85%E7%B7%9A%E9%A7%85%E5%89%8D%E5%81%9C%E7%95%99%E6%89%80%7C%E7%B2%BE%E6%96%87%E9%A4%A8%E6%9B%B8%E5%BA%97%E8%B1%8A%E6%A9%8B%E6%9C%AC%E5%BA%97%7C%E3%83%9C%E3%83%B3.%E5%8D%83%E8%B3%80%7C%E8%B1%8A%E6%A9%8B%E5%B8%82%E3%81%BE%E3%81%A1%E3%81%AA%E3%81%8B%E5%9B%B3%E6%9B%B8%E9%A4%A8%7C%E6%B0%B4%E4%B8%8A%E3%83%93%E3%83%AB

総距離: 113

## Compute Route Matrix

Google Cloud で課金と Routes API を有効にし、下の `API_KEY` にキーを入力してください（または環境変数 `GOOGLE_MAPS_API_KEY` を設定）。このセルの実行は有料 API リクエストを送信します。キーを入力したまま Notebook を公開しないでください。

[Compute Route Matrix の公式ドキュメント](https://developers.google.com/maps/documentation/routes/reference/rest/v2/TopLevel/computeRouteMatrix)

API は訪問順を決めず、地点間の所要時間を返します。今回は固定した出発地・目的地の間で経由地の全順列（9地点なら 9! = 362,880 通り）を比較し、返された時間行列に対して合計時間が最小となる順番を求めます。最近傍法との違いに注意してください。公共交通の時刻表や各地点での滞在時間を考慮する例ではありません。

In [7]:

# ここに API キーを入力
API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "")
if not API_KEY.strip():
    raise ValueError("API_KEY is not set.")

# 出発地と目的地を固定し、既存の Spot をそのまま使用する
matrix_spots = [origin_point] + waypoints + [destination_point]
if len(waypoints) > 9:
    raise ValueError("please limit the number of waypoints to 9 or fewer.")

api_waypoints = []
for spot in matrix_spots:
    if spot.coordinates is None:
        raise ValueError(f"{spot.name}: GPS coordinates are not available.")
    latitude, longitude = spot.coordinates
    api_waypoints: list[dict[str, object]] = []

    for spot in matrix_spots:
        if spot.coordinates is None:
            raise ValueError(f"{spot.name}: GPS coordinates are not available.")
        latitude, longitude = spot.coordinates

        api_waypoints.append(
            {
                "waypoint": {
                    "location": {
                        "latLng": {
                            "latitude": latitude,
                            "longitude": longitude,
                        }
                    }
                }
            }
        )

response = requests.post(
    "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix",
    headers={
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "originIndex,destinationIndex,status,condition,duration,distanceMeters"
        ),
    },
    json={
        "origins": api_waypoints,
        "destinations": api_waypoints,
        "travelMode": "WALK",
    },
    timeout=60,
)
response.raise_for_status()
raw_elements = response.json()
if not isinstance(raw_elements, list):
    raise ValueError("unexpected API response format.")

elements: list[dict[str, object]] = []
for raw_item in cast(list[object], raw_elements):
    if isinstance(raw_item, dict):
        elements.append(cast(dict[str, object], raw_item))

# レスポンスの順序には依存せず、インデックスで行列に格納する
node_count = len(matrix_spots)
duration_matrix = [[math.inf] * node_count for _ in range(node_count)]
road_distance_matrix = [[math.inf] * node_count for _ in range(node_count)]
for element in elements:
    status = element.get("status")
    if isinstance(status, dict):
        status_dict = cast(dict[str, object], status)
        if status_dict.get("code", 0) != 0:
            raise RuntimeError(f"Failed to retrieve route: {status}")

    if element.get("condition") != "ROUTE_EXISTS":
        continue  # 通行できない組み合わせは無限大のままにする

    origin_index = element.get("originIndex")
    destination_index = element.get("destinationIndex")
    duration = element.get("duration")

    if not isinstance(origin_index, int):
        raise ValueError("API response's originIndex is invalid.")
    if not isinstance(destination_index, int):
        raise ValueError("API response's destinationIndex is invalid.")
    if not isinstance(duration, str):
        raise ValueError("API response's duration is invalid.")

    duration_matrix[origin_index][destination_index] = float(
        duration.removesuffix("s")
    )

    distance_meters = element.get("distanceMeters", 0)
    if isinstance(distance_meters, (int, float)):
        road_distance_matrix[origin_index][destination_index] = distance_meters

print(f"Retrieved walking duration for {node_count} locations.")

Retrieved walking duration for 11 locations.


In [8]:
# 出発地・目的地を除いた経由地だけを並べ替える
best_indices = None
best_seconds = math.inf
for order in permutations(range(1, node_count - 1)):
    candidate = (0, *order, node_count - 1)
    seconds = sum(
        duration_matrix[a][b]
        for a, b in zip(candidate, candidate[1:])
    )
    if seconds < best_seconds:
        best_indices = candidate
        best_seconds = seconds

if best_indices is None:
    raise ValueError("No valid route found that visits all locations and reaches the destination.")

matrix_route_spots = [matrix_spots[i] for i in best_indices]
print("徒歩の合計所要時間が最小となるルート：")
for spot in matrix_route_spots:
    print(spot.name)

total_road_distance = sum(
    road_distance_matrix[a][b]
    for a, b in zip(best_indices, best_indices[1:])
)
print(f"合計所要時間（滞在時間を除く）: {best_seconds / 60:.1f} 分")
print(f"このルートの道路距離: {total_road_distance:.0f} m")

matrix_route_url = generate_google_maps_url(
    origin=origin_point,
    destination=destination_point,
    waypoints=matrix_route_spots[1:-1],
    travel_mode=TravelMode.WALKING,
)
print("\nGoogle Maps URL:")
print(matrix_route_url)

徒歩の合計所要時間が最小となるルート：
豊橋駅
むらたのたこやき
豊川堂カルミヤ店
豊橋駅ビルカルミア
駅ビルカフェUNO-UNO
豊鉄市内線駅前停留所
精文館書店豊橋本店
ボン.千賀
豊橋市まちなか図書館
水上ビル
豊橋駅前展望会議室
合計所要時間（滞在時間を除く）: 20.6 分
このルートの道路距離: 1405 m

Google Maps URL:
https://www.google.com/maps/dir/?api=1&origin=%E8%B1%8A%E6%A9%8B%E9%A7%85&destination=%E8%B1%8A%E6%A9%8B%E9%A7%85%E5%89%8D%E5%B1%95%E6%9C%9B%E4%BC%9A%E8%AD%B0%E5%AE%A4&travelmode=walking&waypoints=%E3%82%80%E3%82%89%E3%81%9F%E3%81%AE%E3%81%9F%E3%81%93%E3%82%84%E3%81%8D%7C%E8%B1%8A%E5%B7%9D%E5%A0%82%E3%82%AB%E3%83%AB%E3%83%9F%E3%83%A4%E5%BA%97%7C%E8%B1%8A%E6%A9%8B%E9%A7%85%E3%83%93%E3%83%AB%E3%82%AB%E3%83%AB%E3%83%9F%E3%82%A2%7C%E9%A7%85%E3%83%93%E3%83%AB%E3%82%AB%E3%83%95%E3%82%A7UNO-UNO%7C%E8%B1%8A%E9%89%84%E5%B8%82%E5%86%85%E7%B7%9A%E9%A7%85%E5%89%8D%E5%81%9C%E7%95%99%E6%89%80%7C%E7%B2%BE%E6%96%87%E9%A4%A8%E6%9B%B8%E5%BA%97%E8%B1%8A%E6%A9%8B%E6%9C%AC%E5%BA%97%7C%E3%83%9C%E3%83%B3.%E5%8D%83%E8%B3%80%7C%E8%B1%8A%E6%A9%8B%E5%B8%82%E3%81%BE%E3%81%A1%E3%81%AA%E3%81%8B%E5%9B%B3%E6%9B%B8%E9%A4%A8%7C%E6%B0%B4%E4%B8

ここでの「最小」は API が返した徒歩の時間行列に対するものです。Google Maps のリンクは訪問順を引き継ぎますが、開いた時点で経路を再計算するため、API の結果と一致しないことがあります。モバイルブラウザ等では経由地数の制限が異なるため、今回の9経由地のリンクは PC ブラウザで確認してください。